In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Function to compute circular convolution (time-domain / matrix)
def circular_convolution_direct(x, h):
    N = len(x)
    y = np.zeros(N)
    for n in range(N):
        for m in range(N):
            y[n] += x[m] * h[(n - m) % N]
    return y

def get_circulant_matrix(h):
    N = len(h)
    C = np.zeros((N, N))
    for i in range(N):
        C[:, i] = np.roll(h, i)
    return C

# Function to format complex numbers into HTML/LaTeX representation
def format_complex_html(val):
    re = np.round(val.real, 3)
    im = np.round(val.imag, 3)
    if abs(re) < 1e-10: re = 0.0
    if abs(im) < 1e-10: im = 0.0
    
    if im == 0:
        s = f"{re:g}"
    elif re == 0:
        if im == 1: s = "j"
        elif im == -1: s = "-j"
        else: s = f"{im:g}j"
    else:
        sign = "+" if im > 0 else "-"
        s = f"{re:g} {sign} {abs(im):g}j"
    return s

# Interactive UI elements
x_input = widgets.Text(
    value='1, 2, 3, 4',
    description='x[n]:',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

h_input = widgets.Text(
    value='1, 0, -1, 0',
    description='h[n]:',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

run_button = widgets.Button(
    description="Compute",
    button_style='success',
    layout=widgets.Layout(width='130px', height='32px')
)

output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output(wait=True)
        try:
            x = np.array([float(val.strip()) for val in x_input.value.split(',')])
            h = np.array([float(val.strip()) for val in h_input.value.split(',')])
            
            if len(x) != len(h):
                print("Error: Sequences x[n] and h[n] must have the same length N.")
                return
            
            N = len(x)
            y_dir = circular_convolution_direct(x, h)
            C_h = get_circulant_matrix(h)
            
            print(f"Sequence length N = {N}\n")
            
            # 1. Print Circulant Matrix using HTML tabular format
            html_content = r"<h4>1. Circulant Matrix $\mathbf{C}_h$ (via $h[n]$):</h4>"
            html_content += "<table style='border-collapse: collapse; text-align: center; margin-bottom: 20px; font-family: sans-serif;'>"
            for row in C_h:
                html_content += "<tr>"
                for val in row:
                    s = format_complex_html(val)
                    html_content += f"<td style='border: 1px solid black; padding: 6px 10px;'>${s}$</td>"
                html_content += "</tr>"
            html_content += "</table>"
            
            # 2. Print circular convolution result y[n] right next to the message in math mode
            y_formatted_elements = [format_complex_html(v) for v in y_dir]
            y_math_str = ", \\ ".join(y_formatted_elements)
            html_content += r"""
            <div style='display: flex; align-items: center; gap: 15px; margin-bottom: 20px;'>
                <h4 style='margin: 0;'>2. Circular Convolution Result $y[n]$:</h4>
                <div style='font-size: 1.1em;'>$$y[n] = \begin{bmatrix} """ + y_math_str + r""" \end{bmatrix}$$</div>
            </div>
            """
            
            display(HTML(html_content))
            
            # 3. Visualization with Matplotlib
            fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
            
            axes[0].stem(range(N), x, basefmt=" ")
            axes[0].set_title("Sequence x[n]")
            axes[0].set_xticks(range(N))
            axes[0].grid(True, alpha=0.3)
            
            axes[1].stem(range(N), h, basefmt=" ")
            axes[1].set_title("Sequence h[n]")
            axes[1].set_xticks(range(N))
            axes[1].grid(True, alpha=0.3)
            
            axes[2].stem(range(N), y_dir, basefmt=" ", linefmt='g-', markerfmt='go')
            axes[2].set_title("Convolution y[n]")
            axes[2].set_xticks(range(N))
            axes[2].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Data validity error: {e}")

run_button.on_click(on_button_clicked)

# Layout configuration
inputs_box = widgets.HBox([
    widgets.VBox([x_input, h_input]),
    widgets.VBox([widgets.Label(""), run_button], layout=widgets.Layout(justify_content='flex-end'))
])

instruction_html = widgets.HTML(
    "<h4>Interactive Study of Circular Convolution</h4>"
    "<p>Enter comma-separated values (e.g. <code>1, 2, 3, 4</code>) &nbsp;|&nbsp; <b>Both signals must have the same length.</b></p>"
)

display(widgets.VBox([
    instruction_html,
    inputs_box,
    output
]))